In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import time
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
)

import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd()

# If the notebook is not opened from the project root,
# set ROOT explicitly to the project directory.
if not (ROOT / "backend").exists():
    ROOT = Path(r"E:\Projects\Deepfake detector")

OUTPUT_DIR = ROOT / "experiments" / "benchmark-visualizations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", ROOT)
print("Python/PyTorch device:", DEVICE)
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is unavailable to this PyTorch environment.")

Project root: E:\Projects\Deepfake detector
Python/PyTorch device: cuda
PyTorch version: 2.5.1+cu124
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
CHECKPOINT_EXTENSIONS = {".pth", ".pt", ".ckpt"}

print("=== CHECKPOINTS ===")
for folder in [ROOT / "experiments", ROOT / "backend"]:
    if folder.exists():
        for path in folder.rglob("*"):
            if path.is_file() and path.suffix.lower() in CHECKPOINT_EXTENSIONS:
                print(path.relative_to(ROOT))

print("\n=== CSV MANIFESTS ===")
for folder in [ROOT / "data", ROOT / "experiments"]:
    if folder.exists():
        for path in folder.rglob("*.csv"):
            print(path.relative_to(ROOT))

=== CHECKPOINTS ===
experiments\fft-baseline\cifake-image-v2\cifake-fft-resnet18-best.pt
experiments\fusion-baseline\cifake-image-v2\cifake-fusion-resnet18-best.pt
experiments\rgb-baseline\cifake-rgb-resnet18-best.pt
experiments\rgb-baseline\cifake-image-v2\cifake-rgb-resnet18-best.pt
experiments\rgb-baseline\cifake-image-v2-smoke\cifake-rgb-resnet18-best.pt
experiments\rgb-smoke\cifake-rgb-resnet18-best.pt
experiments\rgb-smoke-balanced\cifake-rgb-resnet18-best.pt

=== CSV MANIFESTS ===
data\metadata\cifake-image-v1\external-test.csv
data\metadata\cifake-image-v1\test.csv
data\metadata\cifake-image-v1\train.csv
data\metadata\cifake-image-v1\val.csv
data\metadata\cifake-image-v2\external-test.csv
data\metadata\cifake-image-v2\test.csv
data\metadata\cifake-image-v2\train.csv
data\metadata\cifake-image-v2\val.csv


In [3]:
MANIFEST_PATH = Path(
    r"E:\Projects\Deepfake detector\data\metadata\cifake-image-v2\test.csv"
)

if MANIFEST_PATH.exists():
    df = pd.read_csv(MANIFEST_PATH)

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    display(df.head())
else:
    print("Manifest not found:", MANIFEST_PATH)
    print("Update MANIFEST_PATH using the inventory from Step 4.")

Rows: 1354
Columns: ['path', 'label_name', 'label', 'group_id', 'sha256', 'width', 'height', 'format', 'split']


,path,label_name,label,group_id,sha256,width,height,format,split
0,data\metadata\cifake-image-v2\training-source\...,real,0,real/382,259bac34cccc1da7c4b67c41ef72b776fe6042c01d3e40...,32,32,JPEG,test
1,data\metadata\cifake-image-v2\training-source\...,real,0,real/1935,89fc6479cf9162d18999e882da42caab8c5f00666cd673...,32,32,JPEG,test
2,data\metadata\cifake-image-v2\training-source\...,real,0,real/3571,9280f4962eb3732a58ab6a43f80974c3db626a6f7663d2...,32,32,JPEG,test
3,data\metadata\cifake-image-v2\training-source\...,real,0,real/2705,1459e2e0e32eba085481780d80b1d388b340af91c55696...,32,32,JPEG,test
4,data\metadata\cifake-image-v2\training-source\...,real,0,real/1794,8bc0c16d0d25d1024e3c02aa30dc7cc927aad2070d4887...,32,32,JPEG,test


In [5]:
def evaluate_binary_predictions(
    y_true,
    y_pred,
    fake_scores,
    dataset_name,
    model_name,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fake_scores = np.asarray(fake_scores)

    if not (len(y_true) == len(y_pred) == len(fake_scores)):
        raise ValueError("Labels, predictions and scores must have equal lengths.")

    if len(y_true) == 0:
        raise ValueError("No samples supplied.")

    if not set(np.unique(y_true)).issubset({0, 1}):
        raise ValueError("Expected labels 0=real and 1=fake.")

    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    metrics = {
        "dataset": dataset_name,
        "model": model_name,
        "samples": len(y_true),
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "real_precision": precision[0],
        "real_recall": recall[0],
        "real_f1": f1[0],
        "fake_precision": precision[1],
        "fake_recall": recall[1],
        "fake_f1": f1[1],
    }

    print(f"{dataset_name} — {model_name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced accuracy: {balanced_accuracy:.4f}")
    print()
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Real", "Fake"],
        zero_division=0,
    ))

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Real", "Fake"],
        yticklabels=["Real", "Fake"],
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{dataset_name} — {model_name} Confusion Matrix")
    plt.tight_layout()

    safe_name = f"{dataset_name}_{model_name}".replace(" ", "_")
    plt.savefig(OUTPUT_DIR / f"{safe_name}_confusion_matrix.png", dpi=160)
    plt.show()

    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, fake_scores)
        metrics["roc_auc"] = auc

        fpr, tpr, _ = roc_curve(y_true, fake_scores)

        plt.plot(fpr, tpr, label=f"ROC-AUC = {auc:.3f}")
        plt.plot([0, 1], [0, 1], "--", label="Chance baseline")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"{dataset_name} — {model_name} ROC Curve")
        plt.legend()
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"{safe_name}_roc.png", dpi=160)
        plt.show()
    else:
        metrics["roc_auc"] = np.nan
        print("ROC-AUC unavailable: both classes are not present.")

    return metrics

In [6]:
# Populate this list only with metrics returned by real model evaluations.
# Each item should be a dictionary returned by evaluate_binary_predictions().

benchmark_results = []

# Example of the intended workflow after real inference:
# result = evaluate_binary_predictions(
#     y_true, y_pred, fake_scores,
#     dataset_name="CIFAKE",
#     model_name="RGB",
# )
# benchmark_results.append(result)

if benchmark_results:
    results_df = pd.DataFrame(benchmark_results)
    display(results_df)

    # Keep each dataset in its own chart because these are different tasks.
    for dataset_name, group in results_df.groupby("dataset"):
        plot_df = group.melt(
            id_vars="model",
            value_vars=["accuracy", "balanced_accuracy"],
            var_name="metric",
            value_name="score",
        )

        sns.barplot(
            data=plot_df,
            x="model",
            y="score",
            hue="metric",
        )
        plt.ylim(0, 1)
        plt.title(f"{dataset_name} — Model Accuracy Comparison")
        plt.ylabel("Score")
        plt.tight_layout()

        safe_name = dataset_name.replace(" ", "_")
        plt.savefig(OUTPUT_DIR / f"{safe_name}_accuracy_comparison.png", dpi=160)
        plt.show()

    results_df.to_csv(OUTPUT_DIR / "benchmark_metrics.csv", index=False)
else:
    print(
        "No benchmark metrics yet. Load the trained checkpoint, "
        "run inference on the correct test split, and append its results."
    )

No benchmark metrics yet. Load the trained checkpoint, run inference on the correct test split, and append its results.
